# Clash Royale Ladder · Neural network template

Predict the winner of a Ladder match from both decks. For local Windows GPU training, select the `Clash Royale GPU (Python 3.10)` kernel, then run the cells in order. See `README.md` for setup instructions.

### 1. Load the data

`X_unformatted` holds 16 card names per match, with the player deck first. The label `y` is `+1` for a player win and `−1` for an opponent win. Use the first 30,000 CSV rows to try the notebook, or set `MAX_ROWS = None` to load the full file if your runtime has enough RAM.

In [1]:
from urllib.request import urlopen

import numpy as np
import pandas as pd

CSV_URL = (
    "https://huggingface.co/datasets/raymond9326/clash-royale-battles/"
    "resolve/main/game_modes/clash_royale_battles_Ladder.csv"
)
MAX_ROWS = 30_000

player_cards = [f"player_card_{i}" for i in range(1, 9)]
opponent_cards = [f"opponent_card_{i}" for i in range(1, 9)]
card_columns = player_cards + opponent_cards
columns = ["battle_id", "battle_time", "player_tag", "opponent_tag", "winner_tag"] + card_columns

with urlopen(CSV_URL, timeout=60) as response:
    battles = pd.read_csv(response, usecols=columns, nrows=MAX_ROWS)

battles["battle_time"] = pd.to_datetime(
    battles["battle_time"], format="%Y%m%dT%H%M%S.%fZ", utc=True, errors="coerce"
)
battles = battles.dropna(subset=columns)
player_won = battles["winner_tag"].eq(battles["player_tag"])
opponent_won = battles["winner_tag"].eq(battles["opponent_tag"])
battles = battles.loc[player_won ^ opponent_won]
battles = battles.drop_duplicates("battle_id").sort_values(["battle_time", "battle_id"])

X_unformatted = battles[card_columns].to_numpy()
y = np.where(battles["winner_tag"].eq(battles["player_tag"]), 1, -1)
del battles

print(f"Raw inputs: {X_unformatted.shape} | Labels: {y.shape}")

Raw inputs: (29999, 16) | Labels: (29999,)


### 2. Format the input

`X` has shape `(matches, 2, 123, 4)`. Each deck has one row per card and four columns in this order.

| Column | Meaning |
| --- | --- |
| 0 | Card present |
| 1 | Evolution equipped |
| 2 | Hero equipped |
| 3 | In-game level divided by 16 |

The historical CSV supplies presence. The other three columns are zero-filled placeholders for future API data, so those zeros carry no information about the actual forms or levels.

In [2]:
# Fixed card order shared by both decks.
# The historical ClashML vocabulary plus Minion Giant.
card_names = [
    "Archer Queen", "Archers", "Arrows", "Baby Dragon", "Balloon", "Bandit",
    "Barbarian Barrel", "Barbarian Hut", "Barbarians", "Bats", "Battle Healer", "Battle Ram",
    "Berserker", "Bomb Tower", "Bomber", "Boss Bandit", "Bowler", "Cannon",
    "Cannon Cart", "Clone", "Dark Prince", "Dart Goblin", "Earthquake", "Electro Dragon",
    "Electro Giant", "Electro Spirit", "Electro Wizard", "Elite Barbarians", "Elixir Collector", "Elixir Golem",
    "Executioner", "Fire Spirit", "Fireball", "Firecracker", "Fisherman", "Flying Machine",
    "Freeze", "Furnace", "Giant", "Giant Skeleton", "Giant Snowball", "Goblin Barrel",
    "Goblin Cage", "Goblin Curse", "Goblin Demolisher", "Goblin Drill", "Goblin Gang", "Goblin Giant",
    "Goblin Hut", "Goblin Machine", "Goblins", "Goblinstein", "Golden Knight", "Golem",
    "Graveyard", "Guards", "Heal Spirit", "Hog Rider", "Hunter", "Ice Golem",
    "Ice Spirit", "Ice Wizard", "Inferno Dragon", "Inferno Tower", "Knight", "Lava Hound",
    "Lightning", "Little Prince", "Lumberjack", "Magic Archer", "Mega Knight", "Mega Minion",
    "Mighty Miner", "Miner", "Mini P.E.K.K.A", "Minion Giant", "Minion Horde", "Minions",
    "Mirror", "Monk", "Mortar", "Mother Witch", "Musketeer", "Night Witch",
    "P.E.K.K.A", "Phoenix", "Poison", "Prince", "Princess", "Rage",
    "Ram Rider", "Rascals", "Rocket", "Ronin", "Royal Delivery", "Royal Ghost",
    "Royal Giant", "Royal Hogs", "Royal Recruits", "Rune Giant", "Skeleton Army", "Skeleton Barrel",
    "Skeleton Dragons", "Skeleton King", "Skeletons", "Sparky", "Spear Goblins", "Spirit Empress",
    "Suspicious Bush", "Tesla", "The Log", "Three Musketeers", "Tombstone", "Tornado",
    "Valkyrie", "Vines", "Void", "Wall Breakers", "Witch", "Wizard",
    "X-Bow", "Zap", "Zappies",
]

card_indices = pd.Categorical(X_unformatted.ravel(), categories=card_names).codes
if (card_indices < 0).any():
    unknown = np.unique(X_unformatted.ravel()[card_indices < 0])
    raise ValueError(f"Card names missing from the catalog: {unknown.tolist()}")

card_indices = card_indices.reshape(-1, 2, 8)
X = np.zeros((len(y), 2, len(card_names), 4), dtype=np.float16)
match_indices = np.arange(len(y))[:, None]

for side in range(2):
    X[match_indices, side, card_indices[:, side], 0] = 1

print(f"Formatted inputs: {X.shape}")

Formatted inputs: (29999, 2, 123, 4)


### 3. Split the matches

Use scikit-learn's `train_test_split` twice to get approximately 80% training, 10% validation and 10% test. Both calls randomly assign matches using seed 47 and keep each input paired with its label.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=47
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, shuffle=True, random_state=47
)
del X_rest, y_rest

print(f"Train: {len(y_train):,} | Validation: {len(y_val):,} | Test: {len(y_test):,}")

Train: 23,999 | Validation: 3,000 | Test: 3,000


### 4. Design and train your model

The Sequential model flattens each match of shape `(2, 123, 4)` and uses one Dense layer with two softmax outputs. Signed labels are converted to one-hot vectors for categorical cross-entropy. Adam uses a learning rate of `0.001`.

In [ ]:
import keras
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import Adam

cnn = keras.Sequential([
    keras.Input(shape=X_train.shape[1:]),
    Dropout(0.2),
    Flatten(),
    Dense(2, activation='softmax'),
])

cnn.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])

y_train_one_hot = keras.utils.to_categorical((y_train == 1).astype(np.int32), num_classes=2)
y_val_one_hot = keras.utils.to_categorical((y_val == 1).astype(np.int32), num_classes=2)

history = cnn.fit(
    X_train,
    y_train_one_hot,
    validation_data=(X_val, y_val_one_hot),
    epochs=10,
    batch_size=256,
)

# Run once after choosing your final model.
# cnn.evaluate(X_test, keras.utils.to_categorical((y_test == 1).astype(np.int32), num_classes=2))

Epoch 1/10
94/94 [==============================] - 24036s 258s/step - loss: 0.6780 - accuracy: 0.5788 - val_loss: 0.6717 - val_accuracy: 0.5913
Epoch 2/10
94/94 [==============================] - 1s 11ms/step - loss: 0.6691 - accuracy: 0.5954 - val_loss: 0.6698 - val_accuracy: 0.5887
Epoch 3/10
94/94 [==============================] - 3s 30ms/step - loss: 0.6662 - accuracy: 0.5973 - val_loss: 0.6692 - val_accuracy: 0.5943
Epoch 4/10
94/94 [==============================] - 1s 13ms/step - loss: 0.6644 - accuracy: 0.6023 - val_loss: 0.6695 - val_accuracy: 0.5907
Epoch 5/10
94/94 [==============================] - 1s 12ms/step - loss: 0.6634 - accuracy: 0.6033 - val_loss: 0.6694 - val_accuracy: 0.5903
Epoch 6/10
94/94 [==============================] - 2s 23ms/step - loss: 0.6627 - accuracy: 0.6053 - val_loss: 0.6703 - val_accuracy: 0.5890
Epoch 7/10
94/94 [==============================] - 1s 14ms/step - loss: 0.6623 - accuracy: 0.6054 - val_loss: 0.6701 - val_accuracy: 0.5893
Epoch 8/1

### 5. Plot the training history

Compare training and validation loss and accuracy across epochs. Run this cell after training to plot the existing `history` without retraining.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

epochs = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')

for ax, metric, title in zip(axes, ['loss', 'accuracy'], ['Loss', 'Accuracy']):
    ax.plot(epochs, history.history[metric], marker='o', label='Training')
    ax.plot(epochs, history.history[f'val_{metric}'], marker='o', label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend()

plt.show()

ModuleNotFoundError: No module named 'matplotlib'